# Zebrafish Gene–Phenotype Knowledge Graph Pipeline (*Danio rerio*)

## Purpose

This notebook implements the **complete end-to-end pipeline** for constructing Gene–Phenotype Knowledge Graph (KG) triples for Zebrafish (*Danio rerio*) from raw ZFIN data. It processes raw phenotype annotations, combines multiple data sources, enriches with NCBI gene descriptions, and produces standardized relation-wise KG triples.

## Pipeline Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | ZFIN All-Gene Phenotypes | Process `phenoGeneCleanData_fish.txt` — phenotypes for all zebrafish genes |
| 2 | ZFIN Human-Ortholog Phenotypes | Process `pheno_fish_R.txt` — phenotypes for genes with human orthologs |
| 3 | ZFIN ID Mapping | Build ZFIN ID → Gene Symbol dictionary from multiple mapping files |
| 4 | Combine & Intermediate Export | Merge both phenotype sets, deduplicate, explode, save intermediate CSV |
| 5 | NCBI Gene Annotation | Load NCBI gene info, build Symbol → Description dictionary |
| 6 | KG Triple Construction | Annotate with descriptions, add metadata, filter unmapped genes |
| 7 | Validation & Final Export | Quality check, deduplicate, export final relation-wise CSV |

## Final Output Schema

| Column | Description |
|--------|-------------|
| `head` | Gene symbol (zebrafish) |
| `relation` | `Gene_Phenotype` |
| `tail` | Phenotype description string |
| `head_type` | `Gene` |
| `relation_type` | (empty — no sub-relation) |
| `tail_type` | `Phenotype` |
| `kg_source` | `ZFIN` |
| `head_id_is` | `NCBI` |
| `tail_id_is` | (empty) |
| `head_detail_name` | NCBI gene description |
| `tail_detail_name` | Same as tail (phenotype string) |
| `species` | `D.rerio` |

## Data Download Instructions

All databases used in this notebook must be downloaded prior to execution.  
Please refer to the **central download instructions document** for detailed steps:

📄 **[Download Instructions — Link to be added]**

### Required Files

| File | Source | Description |
|------|--------|-------------|
| `phenoGeneCleanData_fish.txt` | ZFIN | Phenotype data for all zebrafish genes |
| `pheno_fish_R.txt` | ZFIN | Phenotype data for zebrafish genes with human orthologs |
| `interpro.txt` | ZFIN | InterPro mapping file (ZFIN ID → Symbol) |
| `refseq.txt` | ZFIN | RefSeq mapping file (ZFIN ID → Symbol) |
| `ensembl_1_to_1.txt` | ZFIN | Ensembl 1-to-1 mapping file (ZFIN ID → Symbol) |
| `mappings.txt` | ZFIN | General ZFIN mappings file (ZFIN ID → Symbol) |
| `Danio_rerio.gene_info` | NCBI Gene | Gene annotation file for zebrafish |

---
## Setup

Import required libraries and define all base paths.

In [1]:
import pandas as pd
import numpy as np

In [2]:
your_path_here = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

In [3]:
# =============================================================================
# BASE PATHS — Update these to match your local directory structure
# =============================================================================

# Path to the ZFIN phenotype data files
ZFIN_PHENOTYPE_DIR = f'{your_path_here}zfin/'

# Path to the ZFIN ID mapping files directory
ZFIN_MAPPING_DIR = f'{your_path_here}zfin/'

# Path to the NCBI gene info file for Danio rerio (zebrafish)
NCBI_GENE_INFO_PATH = f'{your_path_here}databases_for_mapping/ncbi/Danio_rerio.gene_info'

# Output paths
# INTERMEDIATE_CSV_PATH = f'{your_path_here}/ZEBRAFISH/GENE_PHENOTYPE/zebrafish_gene_phenotype_Zfin.csv'
FINAL_OUTPUT_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/zfin/zfin_Zebra_GENE_PHENOTYPE_GENE.csv' 

---
## 1. Phenotype of All Zebrafish Genes (`phenoGeneCleanData_fish.txt`)

This section processes the main ZFIN phenotype file containing phenotype annotations for all zebrafish genes.

**Processing steps:**
1. Load the phenotype data and select relevant columns
2. Remove duplicate entries
3. Fill NaN values in structure/process columns with spaces (required for string concatenation)
4. Construct a composite phenotype string by concatenating:
   - Affected Structure/Process 1 (subterm + superterm)
   - Affected Structure/Process 2 (superterm + subterm)
   - Phenotype Keyword Name + Phenotype Tag
5. Group phenotypes by Gene Symbol (comma-separated aggregation)

**Phenotype string format:**  
`{Structure1_subterm} {Structure1_superterm} {Structure2_superterm} {Structure2_subterm} {Keyword} {Tag}`

In [25]:
# =============================================================================
# Load phenoGeneCleanData_fish.txt — phenotype annotations for all zebrafish genes
# =============================================================================

zf_all_pheno_raw = pd.read_csv(ZFIN_PHENOTYPE_DIR + 'phenoGeneCleanData_fish.txt', sep='	')
print(f"Columns in raw file: {list(zf_all_pheno_raw.columns)}")

# Select only the columns needed for phenotype construction
zf_all_pheno_raw = zf_all_pheno_raw[['ID', 'Gene Symbol', 'Gene ID',
             'Affected Structure or Process 1 subterm Name',
             'Affected Structure or Process 1 superterm Name',
             'Affected Structure or Process 2 subterm name',
             'Affected Structure or Process 2 superterm name',
             'Phenotype Keyword ID', 'Phenotype Keyword Name', 'Phenotype Tag']]

# Remove duplicate rows
zf_all_pheno_raw = zf_all_pheno_raw.drop_duplicates()

# Fill NaN values with spaces — required for string concatenation to work correctly
zf_all_pheno_raw['Affected Structure or Process 1 subterm Name'] = zf_all_pheno_raw['Affected Structure or Process 1 subterm Name'].fillna(' ')
zf_all_pheno_raw['Affected Structure or Process 2 subterm name'] = zf_all_pheno_raw['Affected Structure or Process 2 subterm name'].fillna(' ')
zf_all_pheno_raw['Affected Structure or Process 2 superterm name'] = zf_all_pheno_raw['Affected Structure or Process 2 superterm name'].fillna(' ')

# Construct composite phenotype string from multiple annotation fields
zf_all_pheno_raw['Phenotype'] = (zf_all_pheno_raw['Affected Structure or Process 1 subterm Name'] + ' ' +
                                  zf_all_pheno_raw['Affected Structure or Process 1 superterm Name'] + ' ' +
                                  zf_all_pheno_raw['Affected Structure or Process 2 superterm name'] + ' ' +
                                  zf_all_pheno_raw['Affected Structure or Process 2 subterm name'] + ' ' +
                                  zf_all_pheno_raw['Phenotype Keyword Name'] + ' ' +
                                  zf_all_pheno_raw['Phenotype Tag'])

print(f"Total phenotype entries after deduplication: {len(zf_all_pheno_raw):,}")
zf_all_pheno_raw.head()

Columns in raw file: ['ID', 'Gene Symbol', 'Gene ID', 'Affected Structure or Process 1 subterm ID', 'Affected Structure or Process 1 subterm Name', 'Post-composed Relationship ID', 'Post-composed Relationship Name', 'Affected Structure or Process 1 superterm ID', 'Affected Structure or Process 1 superterm Name', 'Phenotype Keyword ID', 'Phenotype Keyword Name', 'Phenotype Tag', 'Affected Structure or Process 2 subterm ID', 'Affected Structure or Process 2 subterm name', 'Post-composed Relationship (rel) ID', 'Post-composed Relationship (rel) Name', 'Affected Structure or Process 2 superterm ID', 'Affected Structure or Process 2 superterm name', 'Fish ID', 'Fish Display Name', 'Start Stage ID', 'End Stage ID', 'Fish Environment ID', 'Publication ID', 'Figure ID']
Total phenotype entries after deduplication: 145,856


,ID,Gene Symbol,Gene ID,Affected Structure or Process 1 subterm Name,Affected Structure or Process 1 superterm Name,Affected Structure or Process 2 subterm name,Affected Structure or Process 2 superterm name,Phenotype Keyword ID,Phenotype Keyword Name,Phenotype Tag,Phenotype,tail
0,575475791,3m76,ZDB-GENE-210324-7,,axis,,,PATO:0001592,increased curvature,abnormal,"increased curvature, abnormal",NaN
1,575475790,3m76,ZDB-GENE-210324-7,,vertebral column,,,PATO:0001592,increased curvature,abnormal,"increased curvature, abnormal",NaN
2,575493708,4m99,ZDB-GENE-210324-8,,axis,,,PATO:0001592,increased curvature,abnormal,"increased curvature, abnormal",NaN
3,575493707,4m99,ZDB-GENE-210324-8,,vertebral column,,,PATO:0001592,increased curvature,abnormal,"increased curvature, abnormal",NaN
4,575264268,aanat2,ZDB-GENE-991019-6,,whole organism,,,PATO:0001997,decreased amount,abnormal,"decreased amount, abnormal",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
153826,575473687,zyg11,ZDB-GENE-041014-17,,cranial cartilage,,,PATO:0000051,morphology,abnormal,"morphology, abnormal",NaN
153827,575473684,zyg11,ZDB-GENE-041014-17,,ceratohyal cartilage,,Meckel's cartilage,PATO:0000040,distance,abnormal,"distance, abnormal",NaN
153828,575473685,zyg11,ZDB-GENE-041014-17,,ceratohyal cartilage,,palatoquadrate cartilage,PATO:0002326,angle,abnormal,"angle, abnormal",NaN
153829,575473681,zyg11,ZDB-GENE-041014-17,,eye,,,PATO:0000587,decreased size,abnormal,"decreased size, abnormal",NaN


In [23]:
# =============================================================================
# Group phenotypes by Gene Symbol — aggregate all phenotypes per gene
# =============================================================================

zf_all_pheno_grouped = zf_all_pheno_raw.groupby('Gene Symbol')['Phenotype'].agg(','.join).reset_index()
zf_pheno_grouped = zf_all_pheno_grouped

print(f"Unique genes with phenotype annotations: {len(zf_pheno_grouped):,}")
zf_pheno_grouped

Unique genes with phenotype annotations: 7,596


,Gene Symbol,Phenotype
0,3m76,"increased curvature, abnormal, increased curv..."
1,4m99,"increased curvature, abnormal, increased curv..."
2,aanat2,"decreased amount, abnormal, decreased duratio..."
3,aars1,"spatial pattern, abnormal, morphology, abnorm..."
4,abca12,"disrupted, abnormal, morphology, abnormal, in..."
...,...,...
7591,zranb3,"decreased amount, abnormal,apoptotic process ..."
7592,zrsr2,"atrophied, abnormal, decreased size, abnormal..."
7593,zuw,"decreased size, abnormal,granule cell decreas..."
7594,zwa,"melanosome spatial pattern, abnormal, decrease..."


---
## 2. Phenotype for Zebrafish Genes with Human Orthologs (`pheno_fish_R.txt`)

This section processes a separate ZFIN file containing phenotype annotations for zebrafish genes with known human orthologs.

**Key difference from Section 1:** This file uses ZFIN Gene IDs (e.g., `ZDB-GENE-000125-4`) rather than Gene Symbols, and includes Entrez IDs for both zebrafish and human orthologs.

In [26]:
# =============================================================================
# Load pheno_fish_R.txt — phenotypes for zebrafish genes with human orthologs
# =============================================================================

zf_pheno_raw = pd.read_csv(ZFIN_PHENOTYPE_DIR + 'pheno_fish_R.txt', sep='	')
print(f"Columns in raw file: {list(zf_pheno_raw.columns)}")
zf_pheno_raw
# Select relevant columns
zf_pheno_raw = zf_pheno_raw[['Gene ID', 'Entrez Zebrafish Gene ID', 'Entrez Human Gene ID',
             'Affected Structure or Process 1 subterm name',
             'Post-Composed Relationship Name',
             'Affected Structure or Process 1 superterm name',
             'Phenotype Quality', 'Phenotype Tag',
             'Affected Structure or Process 2 subterm name',
             'Affected Structure or Process 2 superterm name']]
zf_pheno_raw
# Fill NaN values with spaces for safe string concatenation
zf_pheno_raw['Affected Structure or Process 1 subterm name'] = zf_pheno_raw['Affected Structure or Process 1 subterm name'].fillna(' ')
zf_pheno_raw['Post-Composed Relationship Name'] = zf_pheno_raw['Post-Composed Relationship Name'].fillna(' ')
zf_pheno_raw['Affected Structure or Process 2 superterm name'] = zf_pheno_raw['Affected Structure or Process 2 superterm name'].fillna(' ')
zf_pheno_raw['Affected Structure or Process 2 subterm name'] = zf_pheno_raw['Affected Structure or Process 2 subterm name'].fillna(' ')

# Cast columns to string type to ensure concatenation works
zf_pheno_raw['Affected Structure or Process 1 superterm name'] = zf_pheno_raw['Affected Structure or Process 1 superterm name'].astype(str)
zf_pheno_raw['Post-Composed Relationship Name'] = zf_pheno_raw['Post-Composed Relationship Name'].astype(str)
zf_pheno_raw['Phenotype Quality'] = zf_pheno_raw['Phenotype Quality'].astype(str)
zf_pheno_raw['Phenotype Tag'] = zf_pheno_raw['Phenotype Tag'].astype(str)
zf_pheno_raw
print(f"Total entries: {len(zf_pheno_raw):,}")
zf_pheno_raw

Columns in raw file: ['Gene ID', 'Entrez Zebrafish Gene ID', 'Entrez Human Gene ID', 'ZFIN Gene Symbol', 'Affected Structure or Process 1 subterm OBO ID', 'Affected Structure or Process 1 subterm name', 'Post-Composed Relationship ID', 'Post-Composed Relationship Name', 'Affected Structure or Process 1 superterm OBO ID', 'Affected Structure or Process 1 superterm name', 'Phenotype Keyword OBO ID', 'Phenotype Quality', 'Phenotype Tag', 'Affected Structure or Process 2 subterm OBO ID', 'Affected Structure or Process 2 subterm name', 'Post-Composed Relationship ID.1', 'Post-Composed Relationship Name.1', 'Affected Structure or Process 2 superterm OBO ID', 'Affected Structure or Process 2 superterm name']
Total entries: 62,637


,Gene ID,Entrez Zebrafish Gene ID,Entrez Human Gene ID,Affected Structure or Process 1 subterm name,Post-Composed Relationship Name,Affected Structure or Process 1 superterm name,Phenotype Quality,Phenotype Tag,Affected Structure or Process 2 subterm name,Affected Structure or Process 2 superterm name
0,ZDB-GENE-000125-4,30120,10683,,,somitogenesis,disrupted,abnormal,,
1,ZDB-GENE-000125-4,30120,10683,,,somite specification,disrupted,abnormal,,
2,ZDB-GENE-000125-4,30120,10683,,,hematopoietic progenitor cell differentiation,process quality,abnormal,,
3,ZDB-GENE-000125-4,30120,10683,,,determination of left/right asymmetry in later...,disrupted,abnormal,,
4,ZDB-GENE-000125-4,30120,10683,,,ectoderm development,disrupted,abnormal,,
...,...,...,...,...,...,...,...,...,...,...
62632,ZDB-SNORNAG-201111-1,,727676,,,whole organism,dead,abnormal,,
62633,ZDB-SNORNAG-201111-1,,727676,,,whole organism,decreased life span,abnormal,,
62634,ZDB-SNORNAG-201111-1,,727676,,,intestine,immature,abnormal,,
62635,ZDB-SNORNAG-201111-1,,727676,,,trunk vasculature,disorganized,abnormal,,


In [27]:
# =============================================================================
# Construct composite phenotype string and group by Gene ID
# =============================================================================

zf_pheno_raw['Phenotype'] = (zf_pheno_raw['Affected Structure or Process 1 subterm name'] + ' ' +
                                         zf_pheno_raw['Post-Composed Relationship Name'] + ' ' +
                                         zf_pheno_raw['Affected Structure or Process 1 superterm name'] + ' ' +
                                         zf_pheno_raw['Phenotype Quality'] + ' ' +
                                         zf_pheno_raw['Phenotype Tag'] + ' ' +
                                         zf_pheno_raw['Affected Structure or Process 2 subterm name'])

zf_pheno_grouped = zf_pheno_raw.groupby('Gene ID')['Phenotype'].agg(','.join).reset_index()

print(f"Unique genes (by ZFIN ID) with human-ortholog phenotypes: {len(zf_pheno_grouped):,}")
zf_pheno_raw
zf_pheno_grouped

Unique genes (by ZFIN ID) with human-ortholog phenotypes: 4,611


,Gene ID,Phenotype
0,ZDB-GENE-000125-4,"somitogenesis disrupted abnormal , som..."
1,ZDB-GENE-000201-18,neural crest cell migration disrupted abno...
2,ZDB-GENE-000201-9,"axon guidance disrupted abnormal , olf..."
3,ZDB-GENE-000208-17,neuromast decreased amount abnormal
4,ZDB-GENE-000208-18,uroporphyrinogen decarboxylase activity de...
...,...,...
4606,ZDB-LNCRNAG-131127-227,fourth ventricle hydrocephalic abnormal
4607,ZDB-MIRNAG-041228-10,canonical NF-kappaB signal transduction occurs...
4608,ZDB-MIRNAG-081203-26,filopodium part_of intersegmental vessel incre...
4609,ZDB-MIRNAG-090929-151,hemopoiesis decreased occurrence abnormal ...


---
## 3. Build ZFIN ID → Gene Symbol Mapping Dictionary

The human-ortholog phenotype file uses ZFIN Gene IDs as identifiers. We need Gene Symbols for the final KG triples. This section builds a comprehensive mapping by combining four ZFIN identifier files.

In [8]:
# =============================================================================
# Load and combine multiple ZFIN ID mapping files
# =============================================================================

# Source 1: InterPro annotations (no header in file)
zfin_id_interpro = pd.read_csv(ZFIN_MAPPING_DIR + 'interpro.txt', sep='	', header=None)
zfin_id_interpro = zfin_id_interpro.rename(columns={0: 'ZFIN ID', 2: 'Symbol'})
zfin_id_interpro = zfin_id_interpro[['ZFIN ID', 'Symbol']]

# Source 2: RefSeq cross-references
zfin_id_refseq = pd.read_csv(ZFIN_MAPPING_DIR + 'refseq.txt', sep='	')
zfin_id_refseq = zfin_id_refseq[['ZFIN ID', 'Symbol']]

# Source 3: Ensembl 1-to-1 mappings
zfin_id_ensembl = pd.read_csv(ZFIN_MAPPING_DIR + 'ensembl_1_to_1.txt', sep='	')
zfin_id_ensembl = zfin_id_ensembl[['ZFIN ID', 'Symbol']]

# Source 4: General ZFIN mappings
zfin_id_mappings = pd.read_csv(ZFIN_MAPPING_DIR + 'mappings.txt', sep='	')
zfin_id_mappings = zfin_id_mappings[['ZFIN ID', 'Symbol']]

# Combine all sources and remove duplicates
zfin_id_combined = pd.concat([zfin_id_interpro, zfin_id_refseq, zfin_id_ensembl, zfin_id_mappings], ignore_index=True)
zfin_id_combined = zfin_id_combined.drop_duplicates()

# Create the dictionary
zfin_id_to_symbol = zfin_id_combined.set_index('ZFIN ID')['Symbol'].to_dict()

print(f"Total unique ZFIN ID → Symbol mappings: {len(zfin_id_combined):,}")
print(f"Dictionary size: {len(zfin_id_to_symbol):,} entries")
zfin_id_to_symbol

Total unique ZFIN ID → Symbol mappings: 48,899
Dictionary size: 48,899 entries


{'ZDB-GENE-000112-47': 'ppardb',
 'ZDB-GENE-000125-12': 'igfbp2a',
 'ZDB-GENE-000125-4': 'dlc',
 'ZDB-GENE-000128-11': 'dbx1b',
 'ZDB-GENE-000128-13': 'dbx2',
 'ZDB-GENE-000128-8': 'dbx1a',
 'ZDB-GENE-000201-13': 'anos1b',
 'ZDB-GENE-000201-18': 'pbx4',
 'ZDB-GENE-000201-9': 'anos1a',
 'ZDB-GENE-000208-17': 'calr3a',
 'ZDB-GENE-000208-18': 'urod',
 'ZDB-GENE-000208-20': 'mixl1',
 'ZDB-GENE-000208-21': 'kifc1',
 'ZDB-GENE-000208-23': 'col11a2',
 'ZDB-GENE-000208-25': 'bmp7a',
 'ZDB-GENE-000208-28': 'ccl27a',
 'ZDB-GENE-000208-4': 'zic1',
 'ZDB-GENE-000208-8': 'copg2',
 'ZDB-GENE-000209-3': 'robo1',
 'ZDB-GENE-000209-4': 'robo3',
 'ZDB-GENE-000210-10': 'tpst1',
 'ZDB-GENE-000210-13': 'nccrp1',
 'ZDB-GENE-000210-15': 'btg2',
 'ZDB-GENE-000210-17': 'elavl1b',
 'ZDB-GENE-000210-19': 'rbp4',
 'ZDB-GENE-000210-20': 'cat',
 'ZDB-GENE-000210-21': 'inhbaa',
 'ZDB-GENE-000210-23': 'meis2b',
 'ZDB-GENE-000210-25': 'khdrbs1a',
 'ZDB-GENE-000210-28': 'chico',
 'ZDB-GENE-000210-31': 'vdra',
 'ZDB-GEN

---
## 4. Combine, Deduplicate, Explode & Save Intermediate CSV

Map gene symbols to the human-ortholog data, merge both phenotype datasets, remove duplicate phenotypes per gene, and explode into individual rows. Save the intermediate CSV for record-keeping.

In [9]:
# =============================================================================
# Map ZFIN Gene IDs to Gene Symbols in human-ortholog data
# =============================================================================

zf_pheno_grouped['Gene Symbol'] = zf_pheno_grouped['Gene ID'].map(zfin_id_to_symbol)
zf_hum_ortho_gene_phenotype = zf_pheno_grouped[['Gene Symbol', 'Phenotype']]

unmapped = zf_hum_ortho_gene_phenotype[zf_hum_ortho_gene_phenotype['Gene Symbol'].isna()]
print(f"Human-ortholog genes mapped: {len(zf_hum_ortho_gene_phenotype) - len(unmapped):,}")
print(f"Unmapped ZFIN IDs: {len(unmapped):,}")

In [10]:
# =============================================================================
# Combine both phenotype datasets and deduplicate
# =============================================================================

zf_gene_phenotype_combined = pd.concat([zf_hum_ortho_gene_phenotype, zf_pheno_grouped], ignore_index=True)
zf_gene_phenotype_combined = zf_gene_phenotype_combined.groupby('Gene Symbol')['Phenotype'].agg(','.join).reset_index()

# Remove duplicate phenotype phrases within each gene (preserving order)
def remove_duplicates(phenotype_str):
    """Remove duplicate comma-separated phenotype phrases while preserving order."""
    phrases = phenotype_str.split(',')
    seen = set()
    unique_phrases = []
    for phrase in phrases:
        if phrase not in seen:
            seen.add(phrase)
            unique_phrases.append(phrase)
    return ','.join(unique_phrases)

zf_gene_phenotype_combined['Phenotype'] = zf_gene_phenotype_combined['Phenotype'].apply(remove_duplicates)

# Explode comma-separated phenotypes into individual rows
zf_gene_phenotype_combined['Phenotype'] = zf_gene_phenotype_combined['Phenotype'].str.split(',')
zf_gene_phenotype_combined = zf_gene_phenotype_combined.explode('Phenotype')
zf_gene_phenotype_combined = zf_gene_phenotype_combined.drop_duplicates()

print(f"Combined unique gene-phenotype triples: {len(zf_gene_phenotype_combined):,}")

NameError: name 'zf_hum_ortho_gene_phenotype' is not defined

In [11]:
# =============================================================================
# Save intermediate CSV for record-keeping
# =============================================================================
# Note: DataFrame.rename() with inplace=True is a common pandas pattern;
# inplace parameter is planned for deprecation in future pandas versions

zf_gene_phenotype_combined.rename(columns={'Gene Symbol': 'Head', 'Phenotype': 'Tail'}, inplace=True)
zf_gene_phenotype_combined['Head_type'] = 'Gene'
zf_gene_phenotype_combined['Tail_type'] = 'Phenotype'
zf_gene_phenotype_combined['Relation'] = 'Gene_Phenotype'
zf_gene_phenotype_combined['Relation_source'] = 'ZFin'
zf_gene_phenotype_combined = zf_gene_phenotype_combined[['Head', 'Relation', 'Tail', 'Head_type', 'Tail_type', 'Relation_source']]

# Save intermediate result
# zf_gene_phenotype_combined.to_csv(INTERMEDIATE_CSV_PATH, index=None)
# print(f"Intermediate CSV saved to: {INTERMEDIATE_CSV_PATH}")
print(f"Triples: {len(zf_gene_phenotype_combined):,}")
zf_gene_phenotype_combined

Triples: 62,238


,Head,Relation,Tail,Head_type,Tail_type,Relation_source
0,aanat2,Gene_Phenotype,sleep decreased duration abnormal,Gene,Phenotype,ZFin
0,aanat2,Gene_Phenotype,sleep decreased occurrence abnormal,Gene,Phenotype,ZFin
0,aanat2,Gene_Phenotype,locomotion involved in locomotory behavior...,Gene,Phenotype,ZFin
0,aanat2,Gene_Phenotype,circadian sleep/wake cycle,Gene,Phenotype,ZFin
0,aanat2,Gene_Phenotype,sleep decreased duration abnormal,Gene,Phenotype,ZFin
...,...,...,...,...,...,...
4587,zyg11,Gene_Phenotype,ceratohyal cartilage distance abnormal,Gene,Phenotype,ZFin
4587,zyg11,Gene_Phenotype,ceratohyal cartilage angle abnormal,Gene,Phenotype,ZFin
4587,zyg11,Gene_Phenotype,ceratohyal cartilage flattened abnormal,Gene,Phenotype,ZFin
4587,zyg11,Gene_Phenotype,cranial cartilage morphology abnormal,Gene,Phenotype,ZFin


---
## 5. NCBI Gene Annotation — Build Symbol → Description Dictionary

Load the NCBI gene information file for *Danio rerio* and create a lookup dictionary mapping gene symbols to their functional descriptions.

In [12]:
# =============================================================================
# Load NCBI gene info and build Symbol → Description dictionary
# =============================================================================

ncbi_zebrafish_gene = pd.read_csv(NCBI_GENE_INFO_PATH, sep='	')

ncbi_symbol_to_description = dict(
    zip(ncbi_zebrafish_gene['Symbol'], ncbi_zebrafish_gene['description'])
)

print(f"NCBI genes loaded: {len(ncbi_zebrafish_gene):,}")
print(f"Symbol → Description dictionary size: {len(ncbi_symbol_to_description):,}")

NCBI genes loaded: 51,242
Symbol → Description dictionary size: 42,862


---
## 6. KG Triple Construction — Annotate and Add Metadata

Map NCBI gene descriptions to each gene, filter out unmapped genes, and add all standardized KG metadata columns.

In [13]:
# =============================================================================
# Annotate with NCBI descriptions and add KG metadata
# =============================================================================
# Use the intermediate data (already has Head, Tail, etc.)
# We reference the zebrafish gene symbol from the 'Head' column

gene_phenotype_df = zf_gene_phenotype_combined.copy()

# Map head to lowercase for the final schema
gene_phenotype_df['head'] = gene_phenotype_df['Head']

# Map NCBI gene descriptions
gene_phenotype_df['head_detail_name'] = gene_phenotype_df['head'].map(ncbi_symbol_to_description)

# Filter out genes without NCBI annotations
before_count = len(gene_phenotype_df)
gene_phenotype_df = gene_phenotype_df[~gene_phenotype_df['head_detail_name'].isna()]
after_count = len(gene_phenotype_df)
print(f"Before filtering: {before_count:,}")
print(f"After filtering (genes with NCBI annotations): {after_count:,}")
print(f"Removed: {before_count - after_count:,} triples with unmapped genes")

# Add standardized KG metadata columns
gene_phenotype_df['tail'] = gene_phenotype_df['Tail']
gene_phenotype_df['relation'] = gene_phenotype_df['Relation']
gene_phenotype_df['head_type'] = gene_phenotype_df['Head_type']
gene_phenotype_df['tail_type'] = gene_phenotype_df['Tail_type']
gene_phenotype_df['head_id_is'] = 'NCBI'
gene_phenotype_df['tail_id_is'] = ''
gene_phenotype_df['kg_source'] = 'ZFIN'
gene_phenotype_df['species'] = 'D.rerio'
gene_phenotype_df['relation_type'] = ''
gene_phenotype_df['tail_detail_name'] = gene_phenotype_df['tail']

print(f"Annotated triples: {len(gene_phenotype_df):,}")
gene_phenotype_df

Before filtering: 62,238
After filtering (genes with NCBI annotations): 62,100
Removed: 138 triples with unmapped genes
Annotated triples: 62,100


,Head,Relation,Tail,Head_type,Tail_type,Relation_source,head,head_detail_name,tail,relation,head_type,tail_type,head_id_is,tail_id_is,kg_source,species,relation_type,tail_detail_name
0,aanat2,Gene_Phenotype,sleep decreased duration abnormal,Gene,Phenotype,ZFin,aanat2,arylalkylamine N-acetyltransferase 2,sleep decreased duration abnormal,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,sleep decreased duration abnormal
0,aanat2,Gene_Phenotype,sleep decreased occurrence abnormal,Gene,Phenotype,ZFin,aanat2,arylalkylamine N-acetyltransferase 2,sleep decreased occurrence abnormal,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,sleep decreased occurrence abnormal
0,aanat2,Gene_Phenotype,locomotion involved in locomotory behavior...,Gene,Phenotype,ZFin,aanat2,arylalkylamine N-acetyltransferase 2,locomotion involved in locomotory behavior...,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,locomotion involved in locomotory behavior...
0,aanat2,Gene_Phenotype,circadian sleep/wake cycle,Gene,Phenotype,ZFin,aanat2,arylalkylamine N-acetyltransferase 2,circadian sleep/wake cycle,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,circadian sleep/wake cycle
0,aanat2,Gene_Phenotype,sleep decreased duration abnormal,Gene,Phenotype,ZFin,aanat2,arylalkylamine N-acetyltransferase 2,sleep decreased duration abnormal,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,sleep decreased duration abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4587,zyg11,Gene_Phenotype,ceratohyal cartilage distance abnormal,Gene,Phenotype,ZFin,zyg11,"zyg-11 family member, cell cycle regulator",ceratohyal cartilage distance abnormal,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,ceratohyal cartilage distance abnormal
4587,zyg11,Gene_Phenotype,ceratohyal cartilage angle abnormal,Gene,Phenotype,ZFin,zyg11,"zyg-11 family member, cell cycle regulator",ceratohyal cartilage angle abnormal,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,ceratohyal cartilage angle abnormal
4587,zyg11,Gene_Phenotype,ceratohyal cartilage flattened abnormal,Gene,Phenotype,ZFin,zyg11,"zyg-11 family member, cell cycle regulator",ceratohyal cartilage flattened abnormal,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,ceratohyal cartilage flattened abnormal
4587,zyg11,Gene_Phenotype,cranial cartilage morphology abnormal,Gene,Phenotype,ZFin,zyg11,"zyg-11 family member, cell cycle regulator",cranial cartilage morphology abnormal,Gene_Phenotype,Gene,Phenotype,NCBI,,ZFIN,D.rerio,,cranial cartilage morphology abnormal


---
## 7. Validation, Deduplication & Final Export

Validate data quality, deduplicate by (head, relation, tail) grouping, and export the final relation-wise CSV.

In [14]:
# =============================================================================
# Data quality validation — check for NaN values
# =============================================================================

gene_phenotype_final = gene_phenotype_df

true_nan_count = gene_phenotype_final[['head','relation','tail','head_type','tail_type',
    'head_id_is','tail_id_is','kg_source','species','relation_type',
    'head_detail_name','tail_detail_name']].isna().sum()

print("NaN check on final columns:")
print(true_nan_count)
print(f"Total NaN values: {true_nan_count.sum()}")

NaN check on final columns:
head                0
relation            0
tail                0
head_type           0
tail_type           0
head_id_is          0
tail_id_is          0
kg_source           0
species             0
relation_type       0
head_detail_name    0
tail_detail_name    0
dtype: int64
Total NaN values: 0


In [15]:
# =============================================================================
# Deduplicate by grouping on (head, relation, tail)
# =============================================================================

group_cols = ['head', 'relation', 'tail']

def merge_sources(x):
    """Merge multiple kg_source values with '::' separator."""
    return '::'.join(sorted(set(x.dropna())))

gene_phenotype_dedup = gene_phenotype_final.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Triples before deduplication: {len(gene_phenotype_final):,}")
print(f"Triples after deduplication: {len(gene_phenotype_dedup):,}")
print(f"Duplicates removed: {len(gene_phenotype_final) - len(gene_phenotype_dedup):,}")
gene_phenotype_dedup.head()

Triples before deduplication: 62,100
Triples after deduplication: 62,100
Duplicates removed: 0


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,aanat2,Gene_Phenotype,circadian sleep/wake cycle,Gene,,Phenotype,ZFIN,NCBI,,arylalkylamine N-acetyltransferase 2,circadian sleep/wake cycle,D.rerio
1,aanat2,Gene_Phenotype,locomotion involved in locomotory behavior...,Gene,,Phenotype,ZFIN,NCBI,,arylalkylamine N-acetyltransferase 2,locomotion involved in locomotory behavior...,D.rerio
2,aanat2,Gene_Phenotype,sleep decreased duration abnormal,Gene,,Phenotype,ZFIN,NCBI,,arylalkylamine N-acetyltransferase 2,sleep decreased duration abnormal,D.rerio
3,aanat2,Gene_Phenotype,sleep decreased occurrence abnormal,Gene,,Phenotype,ZFIN,NCBI,,arylalkylamine N-acetyltransferase 2,sleep decreased occurrence abnormal,D.rerio
4,aanat2,Gene_Phenotype,sleep decreased duration abnormal,Gene,,Phenotype,ZFIN,NCBI,,arylalkylamine N-acetyltransferase 2,sleep decreased duration abnormal,D.rerio


In [16]:
FINAL_OUTPUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/zfin/zfin_Zebra_GENE_PHENOTYPE_GENE.csv'

In [17]:
# =============================================================================
# Export the final deduplicated relation-wise KG triples
# =============================================================================

gene_phenotype_dedup.to_csv(FINAL_OUTPUT_PATH, index=None)
print(f"Final output saved to: {FINAL_OUTPUT_PATH}")
print(f"Total triples exported: {len(gene_phenotype_dedup):,}")

Final output saved to: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/zfin/zfin_Zebra_GENE_PHENOTYPE_GENE.csv
Total triples exported: 62,100


---
## Summary

This notebook executed the complete Gene–Phenotype KG pipeline for Zebrafish:

1. **ZFIN phenotype processing** — Processed ~145K raw entries from all-gene phenotypes and ~62K from human-ortholog phenotypes
2. **ID mapping** — Built comprehensive ZFIN ID → Symbol dictionary from 4 sources
3. **Combination & deduplication** — Merged both sets into ~146K unique gene-phenotype pairs
4. **NCBI annotation** — Enriched with gene descriptions, filtered to ~136K annotated triples
5. **Final deduplication** — Grouped by (head, relation, tail) for unique KG triples

### Output Files

| File | Description |
|------|-------------|
| `zfin_Zebra_GENE_PHENOTYPE_GENE.csv` | Final: deduplicated relation-wise KG triples with full metadata |

In [28]:
pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/processed_data/zfin/zfin_Zebra_GENE_PHENOTYPE_GENE.csv')


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,aanat2,Gene_Phenotype,circadian sleep/wake cycle,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,arylalkylamine N-acetyltransferase 2,circadian sleep/wake cycle,D.rerio
1,aanat2,Gene_Phenotype,locomotion involved in locomotory behavior...,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,arylalkylamine N-acetyltransferase 2,locomotion involved in locomotory behavior...,D.rerio
2,aanat2,Gene_Phenotype,sleep decreased duration abnormal,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,arylalkylamine N-acetyltransferase 2,sleep decreased duration abnormal,D.rerio
3,aanat2,Gene_Phenotype,sleep decreased occurrence abnormal,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,arylalkylamine N-acetyltransferase 2,sleep decreased occurrence abnormal,D.rerio
4,aanat2,Gene_Phenotype,sleep decreased duration abnormal,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,arylalkylamine N-acetyltransferase 2,sleep decreased duration abnormal,D.rerio
...,...,...,...,...,...,...,...,...,...,...,...,...
62095,zyg11,Gene_Phenotype,cranial cartilage hypoplastic abnormal,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,"zyg-11 family member, cell cycle regulator",cranial cartilage hypoplastic abnormal,D.rerio
62096,zyg11,Gene_Phenotype,cranial cartilage morphology abnormal,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,"zyg-11 family member, cell cycle regulator",cranial cartilage morphology abnormal,D.rerio
62097,zyg11,Gene_Phenotype,eye decreased size abnormal,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,"zyg-11 family member, cell cycle regulator",eye decreased size abnormal,D.rerio
62098,zyg11,Gene_Phenotype,notochord undulate abnormal,Gene,NaN,Phenotype,ZFIN,NCBI,NaN,"zyg-11 family member, cell cycle regulator",notochord undulate abnormal,D.rerio
